[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/megusto0/pytorch-lab/blob/main/05_pytorch_classification.ipynb)

# 05 - Классификация: nn.Module и nn.Sequential

**Цель.** Решить задачу классификации Iris и сравнить два эквивалентных способа объявления нейросети в PyTorch.

**Результаты обучения:**
- готовить данные для многоклассовой классификации
- использовать CrossEntropyLoss без softmax на выходе
- объявлять архитектуру через nn.Module
- объявлять ту же архитектуру через nn.Sequential
- считать accuracy и строить confusion matrix

## Источники
- [Heaton: PyTorch classification sequence](https://github.com/jeffheaton/app_deep_learning/blob/main/t81_558_class_02_4_pytorch_class_sequence.ipynb)

Импортируем библиотеки. Для матрицы ошибок используем seaborn, который также указан в `requirements.txt`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

%matplotlib inline

torch.manual_seed(42)
np.random.seed(42)

Загрузим Iris, сделаем stratified split 80/20 и стандартизируем признаки. Метки классов оставляем целыми числами для `CrossEntropyLoss`.

In [ ]:
iris = load_iris()
X = iris.data.astype("float32")
y = iris.target.astype("int64")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype("float32")
X_test = scaler.transform(X_test).astype("float32")

train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
test_ds = TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test))
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64)

print("Classes:", iris.target_names)

Модель A описана через собственный класс. Такой подход удобен, когда нужен нестандартный `forward`.

In [ ]:
class IrisModule(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(4, 50)
        self.fc2 = nn.Linear(50, 25)
        self.fc3 = nn.Linear(25, 3)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

model_a = IrisModule()
model_a

Модель B задает ту же архитектуру через `nn.Sequential`. Это компактно для простой прямой цепочки слоев.

In [ ]:
def make_sequential_model():
    return nn.Sequential(
        nn.Linear(4, 50),
        nn.ReLU(),
        nn.Linear(50, 25),
        nn.ReLU(),
        nn.Linear(25, 3),
    )

model_b = make_sequential_model()
model_b

Напишем общую функцию обучения. `CrossEntropyLoss` принимает logits, поэтому softmax в модели не нужен.

In [ ]:
def evaluate_accuracy(model, loader):
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for xb, yb in loader:
            logits = model(xb)
            preds = logits.argmax(dim=1)
            all_preds.extend(preds.numpy())
            all_targets.extend(yb.numpy())
    return accuracy_score(all_targets, all_preds), np.array(all_preds), np.array(all_targets)


def train(model, train_loader, test_loader, epochs=100, lr=1e-2):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_history = []

    for epoch in range(epochs):
        model.train()
        epoch_losses = []
        for xb, yb in train_loader:
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            epoch_losses.append(loss.item())
        loss_history.append(float(np.mean(epoch_losses)))

    accuracy, preds, targets = evaluate_accuracy(model, test_loader)
    return accuracy, loss_history, preds, targets

Обучим обе модели с одинаковым seed. Небольшие различия возможны из-за случайной инициализации, но архитектуры эквивалентны.

In [ ]:
torch.manual_seed(42)
model_a = IrisModule()
accuracy_a, loss_a, preds_a, targets_a = train(model_a, train_loader, test_loader)

torch.manual_seed(42)
model_b = make_sequential_model()
accuracy_b, loss_b, preds_b, targets_b = train(model_b, train_loader, test_loader)

print(f"Accuracy nn.Module: {accuracy_a:.3f}")
print(f"Accuracy nn.Sequential: {accuracy_b:.3f}")

Построим матрицу ошибок для модели `nn.Module`. Диагональные элементы показывают правильно распознанные классы. *

In [ ]:
cm = confusion_matrix(targets_a, preds_a)

plt.figure(figsize=(5, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=iris.target_names,
    yticklabels=iris.target_names,
)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("Confusion matrix")
plt.show()

## Сравнение подходов

`nn.Module` предпочтителен, когда в `forward` нужна нестандартная логика: ветвления, несколько входов или выходов, переиспользование слоев, промежуточные вычисления. Такой класс проще расширять и отлаживать в сложных моделях.

`nn.Sequential` удобен для простой feed-forward архитектуры, где данные проходят через слои строго по очереди. Он короче и хорошо подходит для быстрых прототипов.

> Материал основан на курсе Jeff Heaton T81-558 и книге Maxim Lapan 'Deep Reinforcement Learning Hands-On', глава 3. Адаптировано для учебных целей.